In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

# ۱. خواندن فایل اصلی
file_path = r'second_stage_inputs\G11\dsas_g11_generator_bearings_output.xlsx'
output_filename = r'outputs\G11\dsas_g11_generator_bearings_deviation_monitoring\deviation_monitoring\dsas_g11_generator_bearings_deviation_monitoring_output4.xlsx'
df_raw = pd.read_excel(file_path)

# لیست فیچرها و تارگت‌ها
all_features = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']
# لیست سنسورهایی که تارگت هستند
target_sensors = ['AssetID_9362', 'AssetID_9363', 'AssetID_9364', 'AssetID_9365', 'AssetID_9366', 'AssetID_9367', 'AssetID_9371', 'AssetID_9372', 'AssetID_9373']

# ۲. بارگذاری داده‌ها
df = pd.read_excel(file_path)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(by='date')

df_clean = df.dropna(subset=all_features).copy()
raw_data = df_clean[all_features].values

# ۳. نرمال‌سازی
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(raw_data)

# ۴. ساخت معماری اتوانکودر (همان ساختار قبلی)
input_dim = len(all_features)
input_layer = Input(shape=(input_dim,))
encoded = Dense(16, activation='relu')(input_layer)
latent_space = Dense(8, activation='relu')(encoded)
decoded = Dense(16, activation='relu')(latent_space)
output_layer = Dense(input_dim, activation='sigmoid')(decoded)

autoencoder = Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer=Adam(learning_rate=0.01), loss='mse')

# ۵. آموزش مدل (روی کل داده‌ها برای یادگیری دقیق رفتار سیستم)
print("🚀 در حال یادگیری رفتار سیستم از کل داده‌ها...")
autoencoder.fit(scaled_data, scaled_data, epochs=150, batch_size=32, shuffle=True, verbose=0)

# ۶. محاسبه بازسازی و خطای آن
reconstructed_data = autoencoder.predict(scaled_data)
mse_errors = np.mean(np.power(scaled_data - reconstructed_data, 2), axis=1)

# ۷. محاسبه آستانه ناهنجاری
threshold = np.mean(mse_errors) + (3*np.std(mse_errors))

# ۸. اضافه کردن نتایج به دیتافریم
df_clean['Systematic_Deviation_Index'] = mse_errors
df_clean['Anomaly_Threshold'] = threshold
df_clean['Is_Anomaly'] = df_clean['Systematic_Deviation_Index'] > threshold

# ================= بخش جدید: فیلتر کردن برای یک ماه آخر =================
last_date = df_clean['date'].max()
start_of_last_month = last_date - pd.Timedelta(days=30)
df_last_month = df_clean[df_clean['date'] >= start_of_last_month].copy()
# ======================================================================

# ۹. ذخیره خروجی (فقط دیتای ماه آخر)
os.makedirs(os.path.dirname(output_filename), exist_ok=True)
df_last_month.to_excel(output_filename, index=False)

print("-" * 40)
print(f"✅ تحلیل با موفقیت انجام شد.")
print(f"📅 بازه خروجی: از {df_last_month['date'].min()} تا {df_last_month['date'].max()}")
print(f"🚨 تعداد ناهنجاری در ماه اخیر: {df_last_month['Is_Anomaly'].sum()}")
print(f"📂 فایل (فقط ماه آخر) در مسیر زیر ذخیره شد:\n{output_filename}")

🚀 در حال یادگیری رفتار سیستم از کل داده‌ها...
373/373 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
----------------------------------------
✅ تحلیل با موفقیت انجام شد.
📅 بازه خروجی: از 2026-05-01 20:50:26 تا 2026-05-31 20:30:17
🚨 تعداد ناهنجاری در ماه اخیر: 2
📂 فایل (فقط ماه آخر) در مسیر زیر ذخیره شد:
outputs\G11\dsas_g11_generator_bearings_deviation_monitoring\deviation_monitoring\dsas_g11_generator_bearings_deviation_monitoring_output4.xlsx
